
# Agente Explicador de Regras do Futebol com RAG (LangChain)

## Curso: LangChain — Criando Chatbots Inteligentes com RAG (Foundation)

### Objetivo
Construir um agente conversacional simples utilizando **Retrieval-Augmented Generation (RAG)** para responder perguntas sobre **regras do futebol**, com base em **documentos oficiais**.

---

### Conceitos abordados
- Definição do problema
- Seleção da base de conhecimento
- Preparação dos documentos
- Embeddings e banco vetorial
- Recuperação de contexto (Retriever)
- Integração com LLM
- Testes e validação das respostas



## Dependências

Execute no terminal antes de rodar o notebook:

```bash
pip install langchain langchain-community langchain-openai chromadb pypdf
```


In [ ]:
# Importações básicas
import os

# Loader de documentos PDF
from langchain_community.document_loaders import PyPDFLoader

# Divisão de texto em blocos
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embeddings
from langchain_openai import OpenAIEmbeddings

# Banco vetorial
from langchain_community.vectorstores import Chroma

# LLM
from langchain_openai import ChatOpenAI

#Esse import abaixo é considerado ultrapassado e legado, não recomendado pelo LangChain, mas ainda é possível usar na versão classic
from langchain_classic.chains import RetrievalQA

#Esse é o modelo novo e recomendado
#import langchain


## 1️⃣ Definição do Problema

LLMs possuem conhecimento estático e podem alucinar.
O objetivo aqui é garantir **respostas confiáveis**, conectando o modelo
a documentos oficiais sobre regras do futebol.


In [ ]:

# Caminho do PDF com regras oficiais do futebol
CAMINHO_PDF = r"C:\\Users\\stgab\\codigos_git\\Grupo_2_atividades\\projetos_cientis_dados_biel\\regras_futebol.pdf"  # ajuste o caminho se necessário

# Carrega o PDF
loader = PyPDFLoader(CAMINHO_PDF)
documents = loader.load()

# Quantidade de páginas carregadas
len(documents)


In [ ]:
documents ## O que é ?


## 2️⃣ Preparação dos Documentos

Os documentos precisam ser divididos em pequenos blocos
para facilitar a recuperação de contexto.


In [ ]:

# Divide os documentos em chunks menores
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)

len(chunks)


In [ ]:
chunks ## mostrar 3 pedaços


## 3️⃣ Embeddings e Banco Vetorial

Cada bloco de texto será convertido em vetores semânticos
e armazenado em um banco vetorial.


In [ ]:

# Inicializa embeddings
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=""
)

# Cria o banco vetorial
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_regras_futebol"
)



## 4️⃣ Recuperação de Contexto (Retriever)

O retriever busca os trechos mais relevantes
para cada pergunta do usuário.


In [ ]:

# Cria o retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)



## 5️⃣ Integração com o LLM (RAG)

O contexto recuperado será injetado no prompt
enviado ao modelo de linguagem.


In [ ]:

# Inicializa o modelo de linguagem
llm = ChatOpenAI(
    model="gpt-4o-mini",
    openai_api_key=""
)

# Cria a cadeia RAG
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True
)



## 6️⃣ Testes e Validação

Agora podemos testar perguntas reais
e validar se as respostas estão baseadas nos documentos.


In [ ]:
# Pergunta de teste
pergunta = "Um jogador pode usar a mão para marcar um gol?"

# Executa a pergunta no agente RAG
resposta = qa_chain(pergunta)

print("Pergunta:")
print(pergunta)

print("\nResposta do Agente:")
print(resposta["result"])

print("\nTrechos utilizados como contexto:\n")

for i, doc in enumerate(resposta["source_documents"], start=1):
    print(f"--- Trecho {i} ---")
    print(f"Fonte: {doc.metadata.get('source', 'Documento desconhecido')}")
    print(f"Página: {doc.metadata.get('page', 'N/A')}")
    print("Conteúdo:")
    print(doc.page_content)
    print("\n")


In [ ]:
# Pergunta de teste 2
pergunta = "O que caracteriza a posição de impedimento no futebol?"

# Executa a pergunta no agente RAG
resposta = qa_chain(pergunta)

print("Pergunta:")
print(pergunta)

print("\nResposta do Agente:")
print(resposta["result"])

print("\nTrechos utilizados como contexto:\n")

for i, doc in enumerate(resposta["source_documents"], start=1):
    print(f"--- Trecho {i} ---")
    print(f"Fonte: {doc.metadata.get('source', 'Documento desconhecido')}")
    print(f"Página: {doc.metadata.get('page', 'N/A')}")
    print("Conteúdo:")
    print(doc.page_content)
    print("\n")



## ✅ Conclusão

Este notebook demonstrou, de forma prática:
- Como o RAG melhora a confiabilidade das respostas
- Como conectar LLMs a documentos oficiais
- Como construir um agente conversacional simples com LangChain
